# Learning digits from their sums

The training data are pairs of MNIST images labelled only with the **sum** of their digits. A formula carries the supervision from the sum to the digits (distant supervision).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
import sltn
from sltn import Signature, Interpretation

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

## Data

The first 60 000 MNIST training images become 30 000 pairs, the first 10 000 test images 5 000 pairs. Each pair keeps only the sum of its two labels.

In [ ]:
def make_pairs(train):
    mnist = torchvision.datasets.MNIST("../datasets/", train=train, download=True)
    images = mnist.data.float().unsqueeze(1) / 255.0
    n = len(images) // 2
    return torch.stack([images[:n], images[n:2 * n]], dim=1), (mnist.targets[:n] + mnist.targets[n:2 * n]).float().unsqueeze(-1)


train_images, train_sums = make_pairs(train=True)
test_images, test_sums = make_pairs(train=False)
print(train_images.shape, train_sums.shape, test_images.shape)

## 1 — Signature

- sorts `Image` and `Natnum` (a number);
- a function `digit: Image → Natnum`, the network;
- an infix function `+` and an infix predicate `==` on numbers (Tutorial 1);
- variables `x1`, `x2` for the two images and `s` for the sum.

In [ ]:
sig = Signature("digit_addition")
sig.sort("Image")
sig.sort("Natnum")
sig.function("digit", ["Image"], "Natnum")
sig.function("+", ["Natnum", "Natnum"], "Natnum", infix=True)
sig.predicate("==", ["Natnum", "Natnum"], infix=True)
sig.variable("x1", "Image")
sig.variable("x2", "Image")
sig.variable("s", "Natnum")
print(sig)

## 2 — Knowledge base

```
∀ (x1, x2, s): digit(x1) + digit(x2) == s
```

The quantifier is **diagonal** (Tutorial 2 §3): `x1`, `x2` and `s` hold the two images and the sum of the same pairs, row by row, and are zipped rather than combined in all ways.

In [ ]:
phi = sig.parse("∀ (x1, x2, s): digit(x1) + digit(x2) == s")
phi

## 3 — Model

A small CNN gives a probability $p_k(x)$ for each digit $k$, and `digit` returns the expected digit $\sum_k k\, p_k(x)$, a differentiable number. The predicted class is $\arg\max_k p_k(x)$.

In [ ]:
class DigitCNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Conv2d(1, 16, 3, padding=1), torch.nn.ReLU(), torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(16, 32, 3, padding=1), torch.nn.ReLU(), torch.nn.MaxPool2d(2),
            torch.nn.Flatten(), torch.nn.Linear(32 * 7 * 7, 128), torch.nn.ReLU(),
            torch.nn.Linear(128, 10),
        )

    def probabilities(self, x):
        return torch.softmax(self.net(x), dim=-1)

    def forward(self, x):
        return (self.probabilities(x) * torch.arange(10, device=x.device)).sum(-1, keepdim=True)

## 4 — Interpretation

`+` is ordinary addition, and `a == b` is a smooth equality, $\exp(-|a-b|)$: 1 when the numbers are equal, decreasing with their distance.

In [ ]:
cnn = DigitCNN()
interp = Interpretation(sig).to(device)
interp["Image"] = sltn.Type("Image", shape=(1, 28, 28), axis_names=("channel", "height", "width"))
interp["Natnum"] = sltn.Type("Natnum", shape=(1,))
interp["digit"] = cnn
interp["+"] = lambda a, b: a + b
interp["=="] = lambda a, b: torch.exp(-torch.abs(a - b))
print(interp)

b = torch.linspace(0, 10, 200)
plt.figure(figsize=(6, 2.5))
plt.plot(b, torch.exp(-torch.abs(5 - b)))
plt.xlabel("b"); plt.ylabel("5 == b"); plt.title("Smooth equality with a = 5")
plt.show()

## 5 — Metrics

- **sum accuracy**: the predicted classes of the two images add up to the true sum;
- **digit accuracy**: the predicted class of single test images against their true labels, which were never used.

In [ ]:
mnist_test = torchvision.datasets.MNIST("../datasets/", train=False, download=True)
digit_images = (mnist_test.data.float().unsqueeze(1) / 255.0).to(device)
digit_labels = mnist_test.targets.to(device)


def predict(images):
    return cnn.probabilities(images.to(device)).argmax(-1)


def evaluate():
    interp.eval()
    with torch.no_grad():
        predicted_sums = predict(test_images[:, 0]) + predict(test_images[:, 1])
        sum_acc = (predicted_sums == test_sums.squeeze(-1).to(device)).float().mean().item()
        digit_acc = (predict(digit_images) == digit_labels).float().mean().item()
    return sum_acc, digit_acc

## 6 — Training

The loss is $1 - \text{truth}(\phi)$ on minibatches of 32 pairs.

In [ ]:
optimizer = torch.optim.Adam(interp.parameters(), lr=0.001)

for epoch in range(5):
    interp.train()
    total = 0.0
    batches = torch.randperm(len(train_images)).split(32)
    for idx in batches:
        interp["x1"] = train_images[idx, 0].to(device)
        interp["x2"] = train_images[idx, 1].to(device)
        interp["s"] = train_sums[idx].to(device)
        optimizer.zero_grad()
        loss = 1 - interp(phi).value
        loss.backward()
        optimizer.step()
        total += loss.item()
    sum_acc, digit_acc = evaluate()
    print(f"epoch {epoch} | loss {total / len(batches):.4f} | test sum acc {sum_acc:.3f} | digit acc {digit_acc:.3f}")

## 7 — Results

The confusion matrix of single test digits:

In [ ]:
interp.eval()
with torch.no_grad():
    predicted = predict(digit_images).cpu()
confusion = torch.zeros(10, 10, dtype=torch.long)
confusion.index_put_((mnist_test.targets, predicted), torch.ones_like(predicted), accumulate=True)

plt.figure(figsize=(5, 4))
plt.imshow(confusion, cmap="Blues")
plt.colorbar()
plt.xticks(range(10)); plt.yticks(range(10))
plt.xlabel("predicted digit"); plt.ylabel("true digit")
plt.title("Single-digit confusion matrix")
plt.show()

After 10 epochs both the sum and the digit accuracy are high, close to a classifier trained with digit labels. Most remaining errors are between similar-looking digits.

## Summary

- The only labels are sums; the formula `digit(x1) + digit(x2) == s` turns them into a training signal for the digit classifier.
- A function returning a number (the expected digit) and a smooth equality make the arithmetic differentiable.
- The diagonal quantifier `∀ (x1, x2, s)` matches the images and sums of the same pairs.